# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 06 — Visualizations

**What this notebook does:**
Generates all supplementary and report-ready visualizations.

1. Position gain heatmap — driver cluster × circuit cluster *(key paper figure)*
2. Driver cluster distribution by season
3. Position gain distribution by driver cluster
4. Tyre degradation & strategy by driver cluster
5. Top drivers per cluster by avg position gain
6. 2025 driver cluster assignments
7. MAE by cluster (error analysis)

**Input:**  `data/processed/final_dataset.csv`  
**Output:** `figures/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports ready.')

## Step 1 — Configure Paths & Load Data

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIGURES_DIR  = os.path.join(PROJECT_ROOT, 'figures')

df      = pd.read_csv(os.path.join(PROC_DIR, 'final_dataset.csv'))
circ_df = pd.read_csv(os.path.join(PROC_DIR, 'circuit_features.csv'))

# Fix pit lane starts
pit_mask = df['grid_position'] == 0
df.loc[pit_mask, 'grid_position'] = 20
df.loc[pit_mask, 'position_gain'] = 20 - df.loc[pit_mask, 'finish_position']

df['driver_cluster']  = df['driver_cluster'].fillna(0).astype(int)
df['circuit_cluster'] = df['circuit_cluster'].fillna(0).astype(int)

DRIVER_COLORS = {
    'Aggressive':   '#e10600',
    'Frontrunner':  '#1f77b4',
    'Tyre Manager': '#2ca02c',
    'Backmarker':   '#ff7f0e',
}

train_df = df[df['split'] == 'train'].copy()
test_df  = df[df['split'] == 'test'].copy()

print(f'✅ final_dataset: {df.shape}')
print(f'   Train: {len(train_df)} | Test: {len(test_df)}')
print(f'   Seasons: {sorted(df["season"].unique().tolist())}')
print(f'   Driver clusters present: {sorted(df["driver_cluster_name"].dropna().unique().tolist())}')

## Plot 1 — Position Gain Heatmap: Driver Cluster × Circuit Cluster

In [ ]:
pivot = train_df.groupby(['driver_cluster_name', 'circuit_cluster_name'])['position_gain'].mean().unstack()

row_order = [r for r in ['Frontrunner', 'Aggressive', 'Tyre Manager', 'Backmarker'] if r in pivot.index]
col_order = [c for c in ['High-Speed', 'High-Degradation', 'Outlier'] if c in pivot.columns]
pivot = pivot.loc[row_order, col_order]

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)

ax.set_xticks(range(len(col_order)))
ax.set_xticklabels(col_order, fontsize=12)
ax.set_yticks(range(len(row_order)))
ax.set_yticklabels(row_order, fontsize=12)
ax.set_xlabel('Circuit Cluster', fontsize=12, labelpad=10)
ax.set_ylabel('Driver Cluster', fontsize=12, labelpad=10)
ax.set_title('Mean Position Gain: Driver–Circuit Compatibility\n(Training Data 2019–2024)',
             fontweight='bold', fontsize=13)

counts = train_df.groupby(['driver_cluster_name', 'circuit_cluster_name']).size().unstack()
counts = counts.reindex(index=row_order, columns=col_order)

for i in range(len(row_order)):
    for j in range(len(col_order)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.2f}', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white' if abs(val) > 1.2 else 'black')
            n = counts.values[i, j] if not np.isnan(counts.values[i, j]) else 0
            ax.text(j, i + 0.35, f'n={int(n)}', ha='center', va='center', fontsize=8, color='gray')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('Mean Position Gain', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'viz_compatibility_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/viz_compatibility_heatmap.png')
print(pivot.round(3).to_string())

## Plot 2 — Driver Cluster Distribution by Season

In [ ]:
season_cluster = (
    train_df.groupby(['season', 'driver_cluster_name'])
    .size().unstack(fill_value=0)
)
season_cluster_pct = season_cluster.div(season_cluster.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(season_cluster_pct))

for cluster_name in ['Frontrunner', 'Aggressive', 'Tyre Manager', 'Backmarker']:
    if cluster_name in season_cluster_pct.columns:
        vals = season_cluster_pct[cluster_name].values
        ax.bar(season_cluster_pct.index.astype(str), vals, bottom=bottom,
               label=cluster_name, color=DRIVER_COLORS[cluster_name], edgecolor='white', linewidth=0.5)
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 5:
                ax.text(i, b + v/2, f'{v:.0f}%', ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottom += vals

ax.set_title('Driver Cluster Distribution by Season', fontweight='bold', fontsize=13)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('% of Driver-Race Entries', fontsize=11)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'viz_cluster_by_season.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/viz_cluster_by_season.png')

## Plot 3 — Position Gain Distribution by Driver Cluster

In [ ]:
cluster_order = [c for c in ['Frontrunner', 'Aggressive', 'Tyre Manager', 'Backmarker']
                 if c in train_df['driver_cluster_name'].unique()]

fig, axes = plt.subplots(1, len(cluster_order), figsize=(16, 5), sharey=True)

for ax, name in zip(axes, cluster_order):
    data = train_df[train_df['driver_cluster_name'] == name]['position_gain']
    ax.hist(data, bins=30, color=DRIVER_COLORS[name], edgecolor='none', alpha=0.85)
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5, label=f'mean={data.mean():.2f}')
    ax.axvline(0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)
    ax.set_title(name, fontweight='bold', fontsize=11, color=DRIVER_COLORS[name])
    ax.set_xlabel('Position Gain', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    ax.text(0.97, 0.97, f'n={len(data)}', transform=ax.transAxes, ha='right', va='top', fontsize=8, color='gray')

axes[0].set_ylabel('Count', fontsize=10)
fig.suptitle('Position Gain Distribution by Driver Cluster', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'viz_position_gain_by_cluster.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/viz_position_gain_by_cluster.png')

## Plot 4 — Tyre Strategy by Driver Cluster

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
strategy_features = [
    ('tyre_degradation_slope', 'Tyre Degradation Slope'),
    ('avg_stint_length',       'Avg Stint Length (laps)'),
    ('number_of_pit_stops',    'Number of Pit Stops'),
]

for ax, (feat, label) in zip(axes, strategy_features):
    group_data   = [train_df[train_df['driver_cluster_name'] == n][feat].dropna() for n in cluster_order]
    group_colors = [DRIVER_COLORS[n] for n in cluster_order]

    bp = ax.boxplot(group_data, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], group_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_xticks(range(1, len(cluster_order) + 1))
    ax.set_xticklabels(cluster_order, fontsize=9, rotation=15)
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Strategy Features by Driver Cluster (Training Data)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'viz_strategy_by_cluster.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/viz_strategy_by_cluster.png')

## Plot 5 — Top Drivers per Cluster

In [ ]:
driver_summary = (
    train_df.groupby(['driver_id', 'driver_cluster_name'])
    .agg(avg_pos_gain=('position_gain', 'mean'), races=('round', 'count'))
    .reset_index()
)
driver_summary = driver_summary[driver_summary['races'] >= 10]

fig, axes = plt.subplots(1, len(cluster_order), figsize=(18, 6))

for ax, name in zip(axes, cluster_order):
    subset = (
        driver_summary[driver_summary['driver_cluster_name'] == name]
        .sort_values('avg_pos_gain', ascending=True).tail(8)
    )
    colors_bar = ['#2ca02c' if v >= 0 else '#e10600' for v in subset['avg_pos_gain']]
    bars = ax.barh(subset['driver_id'], subset['avg_pos_gain'], color=colors_bar, edgecolor='none', alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(name, fontweight='bold', fontsize=10, color=DRIVER_COLORS[name])
    ax.set_xlabel('Avg Position Gain', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, subset['avg_pos_gain']):
        ax.text(val + (0.05 if val >= 0 else -0.05), bar.get_y() + bar.get_height()/2,
                f'{val:+.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=7.5)

fig.suptitle('Top Drivers per Cluster — Avg Position Gain (Training 2019–2024)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'viz_top_drivers_per_cluster.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: figures/viz_top_drivers_per_cluster.png')

## Plot 6 — 2025 Driver Cluster Assignments

In [ ]:
test_driver_clusters = (
    test_df.groupby(['driver_id', 'driver_cluster_name'])
    .size().reset_index(name='count')
    .sort_values(['driver_cluster_name', 'count'], ascending=[True, False])
)

print('2025 Driver Cluster Assignments:')
print(test_driver_clusters[['driver_id', 'driver_cluster_name']].drop_duplicates().to_string(index=False))

cluster_counts_2025 = test_driver_clusters.groupby('driver_cluster_name')['driver_id'].apply(list)
print('\nBy cluster:')
for cluster, drivers in cluster_counts_2025.items():
    print(f'  {cluster}: {sorted(set(drivers))}')

## Summary of Saved Figures

In [ ]:
print('📊 Figures saved to figures/:')
expected = [
    'driver_cluster_selection.png',
    'circuit_cluster_selection.png',
    'cluster_pca.png',
    'model_comparison.png',
    'predictions_vs_actual.png',
    'shap_summary.png',
    'shap_bar.png',
    'viz_compatibility_heatmap.png',
    'viz_cluster_by_season.png',
    'viz_position_gain_by_cluster.png',
    'viz_strategy_by_cluster.png',
    'viz_top_drivers_per_cluster.png',
]
for f in expected:
    path = os.path.join(FIGURES_DIR, f)
    print(f'  {"✅" if os.path.exists(path) else "❌"} {f}')